# Aqua API — Integration test (existing revision)

Same flow as `train_predict_demo.ipynb`, but reuses an existing version + revision in the API instead of creating fresh ones each run. Intended for repeatable integration testing — keeps repeated runs from accumulating throwaway data.

1. **Authenticate** with credentials loaded from `.env` (via `python-dotenv`).
2. **Look up the reference revision** to align against (NIV, revision id `1592`).
3. **Trigger training** for all available apps against the existing target (`VERSION_ID=1291`, `REVISION_ID=2023`).
4. **Poll training status** until every job reaches a terminal state.
5. **Inspect training results** — interleaved per-vref scores from each app.
6. **Run `/predict`** on a sample verse pair using the artifacts you just trained.

Set the following in your `.env` (or export them in the parent shell) before running:

```
AQUA_USERNAME=...
AQUA_PASSWORD=...
```

> The Aqua API uses the [vref convention](https://github.com/BibleNLP/ebible/blob/main/metadata/vref.txt) — every revision is a 41,899-line text file where line N corresponds to canonical verse reference N.

## What each app does

The training / predict fan-out covers five apps. Each scores a `(source_text, target_text)` pair on a different dimension; the design intent is that they're complementary rather than redundant.

| App | What it does | Strength |
| --- | --- | --- |
| **semantic-similarity** | Compares sentence-pair embeddings from a fine-tuned model. | Catches meaning drift even when the two sides share no surface vocabulary (e.g. a paraphrase that lost the actual meaning). |
| **tfidf** | Builds a [TF-IDF](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) + truncated-SVD encoder over the corpus and returns the nearest-neighbour verses to a query — i.e. the verses whose wording is most similar. | Useful when reviewing a verse: surfaces other places in the corpus that read similarly, so the translator can check consistency or pull comparable renderings. |
| **word-alignment** | Word-level alignment between source and target. With `use_eflomal=true` it uses [eflomal](https://github.com/robertostling/eflomal) (statistical alignment); otherwise FastAlign. | Surfaces added words, omitted words, and systematic mistranslations of specific terms. |
| **ngrams** | Mines frequent multi-word phrases from the training corpus, so translators can compare translation consistency of those phrases. | Catches verses that are missing the recurring phrases your translation team usually uses. |
| **agent-critique** *(likely to be renamed `ai-notes`)* | LLM-driven pipeline. Produces structured per-verse issues (omissions, additions, replacements, severity), an automatic back-translation, lexeme cards, and a grammar sketch. | Surfaces semantic and theological issues a statistical model would miss, plus the back-translation / lexeme / grammar artefacts give human reviewers context to work from. Most expensive of the five — call it via `/predict` on the verses you actually want inspected, not over the whole training set. |

You can fan out to a subset by passing `apps=[...]` on `/train` or `/predict`. Omit it to run all five.

In [ ]:
import json
import os
import time
from pathlib import Path

import requests
from dotenv import load_dotenv


def check(resp):
    """Like ``resp.raise_for_status()`` but surfaces the response body so a
    422 from Pydantic actually tells you which field it didn't like."""
    if not resp.ok:
        raise RuntimeError(
            f"{resp.request.method} {resp.url} -> "
            f"{resp.status_code}: {resp.text}"
        )
    return resp


# Load AQUA_USERNAME / AQUA_PASSWORD from `.env` at the repo root, falling
# back to whatever is already in the parent shell.
load_dotenv()

## 0. Configure

In [ ]:
# Choose the environment you want to hit.
BASE_URL = "https://cp3by92k8p.us-east-1.awsapprunner.com"  # development
# BASE_URL = "https://tmv9bz5v4q.us-east-1.awsapprunner.com"  # production

# All v3 routes are reachable at /latest/ as well — we use /latest/ here.
PREFIX = "latest"

USERNAME = os.environ["MARK_API_USERNAME"]
PASSWORD = os.environ["MARK_API_PASSWORD"]

# Existing target to train + predict against. Created once in the dev
# environment; integration runs reuse these IDs rather than re-uploading.
VERSION_ID = 1291
REVISION_ID = 2023

# Reference revision to align against. 1592 is the NIV in the dev DB.
REFERENCE_REVISION_ID = 1592

## 1. Authenticate

`POST /latest/token` with form-encoded credentials returns a Bearer token. Every subsequent request includes that token in the `Authorization` header.

In [ ]:
resp = check(requests.post(
    f"{BASE_URL}/{PREFIX}/token",
    data={"username": USERNAME, "password": PASSWORD},
))
TOKEN = resp.json()["access_token"]
AUTH = {"Authorization": f"Bearer {TOKEN}"}
print(f"Got token (length: {len(TOKEN)}) for {USERNAME}")

## 2. Resolve the reference (NIV) version_id

Both `/train` and `/predict` are keyed on **versions**. By default the API picks the latest non-deleted revision under the version when training, so a fresh upload to the same version automatically becomes the next training input.

If you need to pin training to a specific revision (e.g. retraining against an older draft for comparison), pass `source_revision_id` / `target_revision_id` instead — see the next cell.

We need the NIV's `bible_version_id` for the train + predict steps below. If you already know it, you can hardcode `REFERENCE_VERSION_ID` directly. Otherwise, the lookup below scans your accessible revisions for `id == REFERENCE_REVISION_ID`.

In [ ]:
resp = check(requests.get(f"{BASE_URL}/{PREFIX}/revision", headers=AUTH))
revisions = resp.json()

ref = next((r for r in revisions if r["id"] == REFERENCE_REVISION_ID), None)
if ref is None:
    raise RuntimeError(
        f"Revision {REFERENCE_REVISION_ID} is not in your accessible-revisions list. "
        "Ask your admin to grant your group access to the NIV reference version, "
        "or set REFERENCE_VERSION_ID directly below."
    )
REFERENCE_VERSION_ID = ref["bible_version_id"]
print(
    f"Reference (revision {REFERENCE_REVISION_ID}) lives under "
    f"version_id {REFERENCE_VERSION_ID}"
)

## 3. Trigger training

`POST /latest/train` queues training jobs for every app at once and returns a `session_id` you can poll. Pass `apps=[...]` to restrict to specific apps; omit it to train all of them.

The pair is identified by **`source_version_id` / `target_version_id`** — the API resolves each side to its latest non-deleted revision. To pin a side to a specific revision (e.g. retrain against an older draft), swap that side's `*_version_id` for `*_revision_id`; you can mix the two across sides.

`use_eflomal=true` opts the word-alignment app into the eflomal alignment path.

In [ ]:
train_payload = {
    "source_version_id": REFERENCE_VERSION_ID,    # NIV reference (latest revision)
    "target_version_id": VERSION_ID,              # your translation (latest revision)
    "options": {"use_eflomal": True},             # Will probably become default soon
    # To pin a side to a specific revision instead of "latest under the version":
    # "source_revision_id": REFERENCE_REVISION_ID,
    # "target_revision_id": REVISION_ID,
    # "apps": ["semantic-similarity", "word-alignment"]  # optional subset
}

resp = check(requests.post(
    f"{BASE_URL}/{PREFIX}/train",
    json=train_payload,
    headers=AUTH,
))
training_response = resp.json()
SESSION_ID = training_response["session_id"]
print(f"Session id: {SESSION_ID}")
for job in training_response["training_jobs"]:
    print(f"  job {job['id']:>4}  {job['type']:<22}  status={job['status']}")

## 4. Poll training status

`GET /latest/train/status?session_id=...` returns the same payload shape as `/train`, so you can watch each job tick from `queued` → `running` → `finished` (or `failed`).

Typical durations on dev:

| App | Typical |
| --- | --- |
| semantic-similarity | ~20-40 min |
| tfidf | ~1–2 min |
| ngrams | ~3 min |
| word-alignment (eflomal) | ~10–15 min |
| agent-critique | ~10-15 min |

In [ ]:
TERMINAL = {"finished", "failed", "cancelled"}
POLL_INTERVAL = 30  # seconds

while True:
    resp = check(requests.get(
        f"{BASE_URL}/{PREFIX}/train/status",
        params={"session_id": SESSION_ID},
        headers=AUTH,
    ))
    jobs = resp.json()["training_jobs"]

    print(f"[{time.strftime('%H:%M:%S')}]")
    for job in jobs:
        pct = (
            f"{job['percent_complete']:>5.1f}%"
            if job.get("percent_complete") is not None
            else "   ?  "
        )
        detail = job.get("status_detail") or ""
        print(
            f"  {job['type']:<22}  {job['status']:<9}  {pct}  {detail}"
        )

    if all(j["status"] in TERMINAL for j in jobs):
        break
    time.sleep(POLL_INTERVAL)

print("\nAll jobs terminal.")

## 5. Inspect training results

`GET /latest/train/status/{session_id}/results` returns per-verse rows from every completed app, interleaved by `vref`. You can filter by `book` / `chapter` / `verse` and paginate with `page` + `page_size`.

Conceptually these results are `/predict` run over **the whole training text** — every app scores every verse you uploaded against the reference, so you get a free pre-scored pass over your translation as a side effect of training. That holds for `semantic-similarity`, `tfidf`, `word-alignment`, and `ngrams`.

The exception is **`agent-critique`**: running it on the entire submitted text would be prohibitively expensive (it's an LLM-driven critique loop). Training only fits the agent artifacts; it does not return per-verse agent results here. To get agent-critique scores for specific verses, call `/predict` (next section) with `apps=["agent-critique"]` and the verses you actually want critiqued.

We filter to a single chapter (Luke 1) just to keep the preview small.

In [ ]:
resp = check(requests.get(
    f"{BASE_URL}/{PREFIX}/train/status/{SESSION_ID}/results",
    params={"book": "LUK", "chapter": 1, "page": 1, "page_size": 10},
    headers=AUTH,
))
results = resp.json()
print(json.dumps(results, indent=2))

## 6. Run `/predict` on a sample verse pair

Once training is done, `/predict` scores arbitrary `(source_text, target_text)` pairs against the trained artifacts. The expectation is that each pair is either a **brand-new draft verse** that was not in the training data, or a **revised version of a verse** that was — so you can score quality after each translator edit without retraining.

Throughout the API the convention is **`source` = reference language (what you're translating *from*), `target` = the translation being scored (what you're translating *to*)**. Here that means `source_text` / `source_version_id` is the NIV English side, and `target_text` / `target_version_id` is the existing target translation.

- `pairs` — list of `{vref, source_text, target_text}`. **Scoring runs on the strings**: ngrams checks overlap with the trained ngram set, semantic-similarity compares text embeddings, and so on. `vref` is an optional convenience identifier — the API echoes it back alongside each pair's results so you can match scores to verses on the client side, but it does not drive scoring.
- `source_version_id` / `target_version_id` — the version pair the artifacts were trained under. Both `/train` and `/predict` are **keyed on versions**: training artifacts are shared across every revision within a version (so a fresh revision of an in-progress translation reuses the same trained models without retraining), but they don't leak across versions — two different versions in the same language are isolated, and you'll need to train each one separately.
- `apps` — optional list to fan out to a subset.
- `include_translation` / `include_critique` — opt the agent app into its heavier LLM passes. These don't block the synchronous response: the fast pieces of the agent (lexeme cards, grammar sketch, language profiles) come back immediately, and the slow translation/critique work runs in the background. The response includes a `job` handle you poll at `/predict/jobs/{id}` to pick up the slow result. See the next section for the polling code.

Below we score two pairs at GEN 1:1: the first uses the NIV English source paired with the actual target rendering of GEN 1:1 (a known-good match), the second pairs the English GEN 1:2 source with the target's GEN 1:3 text (deliberately wrong: same language, different verse) so you can see the score collapse.


In [ ]:
# Fetch the actual GEN 1:1 and GEN 1:3 text from the existing target
# revision via /verse — keeps the predict pair grounded in the same
# corpus the training artifacts were fit on.
def get_verse(revision_id: int, book: str, chapter: int, verse: int) -> str:
    resp = check(requests.get(
        f"{BASE_URL}/{PREFIX}/verse",
        params={
            "revision_id": revision_id,
            "book": book,
            "chapter": chapter,
            "verse": verse,
        },
        headers=AUTH,
    ))
    rows = resp.json()
    if not rows:
        raise RuntimeError(f"No verse text for {book} {chapter}:{verse} in revision {revision_id}")
    return rows[0]["text"]


target_gen_1_1 = get_verse(REVISION_ID, "GEN", 1, 1)
target_gen_1_3 = get_verse(REVISION_ID, "GEN", 1, 3)
print(f"GEN 1:1 (target): {target_gen_1_1!r}")
print(f"GEN 1:3 (target, used as the bad pair): {target_gen_1_3!r}")

# A reasonable English rendering of GEN 1:1 / 1:2 to use as the source side.
ENGLISH_GEN_1_1 = (
    "In the beginning God created the heavens and the earth."
)
ENGLISH_GEN_1_2 = (
    "Now the earth was formless and void, and darkness was over the surface of the deep. "
    "And the Spirit of God was hovering over the surface of the waters."
)

# Stash the slow-path job id in a known sentinel so the polling cell
# below stays restart-safe even if you re-run it without re-running this
# cell. The real value is set after the predict response below.
JOB_ID = None

predict_payload = {
    "pairs": [
        {
            "vref": "GEN 1:1",
            "source_text": ENGLISH_GEN_1_1,
            "target_text": target_gen_1_1,
        },
        {
            "vref": "GEN 1:2",
            "source_text": ENGLISH_GEN_1_2,
            # Deliberately wrong: target is GEN 1:3's text under English GEN 1:2,
            # so every app should score it low.
            "target_text": target_gen_1_3,
        },
    ],
    "source_version_id": REFERENCE_VERSION_ID,  # NIV
    "target_version_id": VERSION_ID,             # existing target translation
    # "apps": ["semantic-similarity", "ngrams"],  # optional subset
    # Opt the agent into the heavier LLM passes. Both default to False.
    # Setting either to True does NOT make the call block — the slow
    # work is spawned in the background and the response carries a
    # `job` handle you poll for the result (see section 6b below).
    # `include_critique` requires `include_translation=True`.
    "include_translation": True,
    # "include_critique": True,
}

resp = check(requests.post(
    f"{BASE_URL}/{PREFIX}/predict",
    json=predict_payload,
    headers=AUTH,
))
predictions = resp.json()

# Save the full response to a file so it's easy to inspect later.
output_path = Path("predict_output.json")
output_path.write_text(
    json.dumps(predictions, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Saved full predict response to {output_path.resolve()}")

# Per-app status summary so the cell is still informative without
# dumping the whole payload.
for app_name, result in predictions.get("results", {}).items():
    status_str = result.get("status", "?")
    note = ""
    if status_str == "ok":
        n_pairs = len(result.get("data", {}).get("pairs", []))
        note = f"({n_pairs} pair(s) returned)"
    elif result.get("error"):
        note = f"error: {result['error']}"
    print(f"  {app_name:<20} {status_str:<6} {note}")

# If we asked for translation or critique, the response carries a job
# handle pointing at the polling endpoint. Stash the id for the next cell.
JOB_ID = (predictions.get("job") or {}).get("id")
if JOB_ID:
    print(f"\nSpawned slow agent path: job_id={JOB_ID}")
    print(f"Poll at: {predictions['job']['poll_url']}")
else:
    print("\nNo slow-path job — translation/critique were not requested.")


## 6b. Poll the slow agent path for translation / critique

When `/predict` was called with `include_translation=True` (and/or `include_critique=True`), the response above includes a `job` handle. The slow LLM passes can take a few minutes for a chapter-sized batch — far longer than the 2-minute API request limit — so they run in the background and you fetch them via `GET /predict/jobs/{id}`.

The polling endpoint returns one of three statuses:

- `running` — not done yet. The response carries a `Retry-After` header (currently 10 seconds) suggesting a polite poll cadence.
- `complete` — `pairs` is populated with `translation` (and `critique` if you asked for it) for each submitted pair, in the same order they were submitted. Every pair echoes back its original `vref`, `source_text`, and `target_text`, so callers that submit without a `vref` can still match results to inputs by index.
- `failed` — the slow Modal call raised. The `error` field carries a short reason; the synchronous `/predict` response above is unaffected.

Jobs are scoped to the user that created them — only that account (or an admin) can read the result. Job records are kept in the database and survive container restarts, so you can poll from a different process if you need to.


In [ ]:
POLL_INTERVAL = 10  # the API also advertises this via Retry-After
JOB_TERMINAL = {"complete", "failed"}

if not JOB_ID:
    print("Skipping — no job to poll (set include_translation=True above to try this).")
else:
    while True:
        resp = check(requests.get(
            f"{BASE_URL}/{PREFIX}/predict/jobs/{JOB_ID}",
            headers=AUTH,
        ))
        job = resp.json()
        print(f"[{time.strftime('%H:%M:%S')}] status={job['status']}")
        if job["status"] in JOB_TERMINAL:
            break
        # The Retry-After header is the API's preferred cadence; fall
        # back to POLL_INTERVAL if the proxy strips it.
        time.sleep(int(resp.headers.get("Retry-After", POLL_INTERVAL)))

    if job["status"] == "complete":
        print(f"\nGot {len(job['pairs'])} pair(s):\n")
        for p in job["pairs"]:
            label = p.get("vref") or f"(no vref) target={p['target_text'][:30]!r}"
            t = (p.get("translation") or {}).get("literal", "<no translation>")
            print(f"  {label}: {t[:120]}")
    else:
        print(f"\nJob failed: {job.get('error')!r}")
